# Airbnb NYC 2019 — Data Understanding

## Purpose

This notebook performs the initial inspection of Airbnb NYC 2019 dataset before Exploratory Data Analysis (EDA) phase, data preparation and machine-learning modelling.

The objectives are to:

- Understanding the size and structure of the dataset;
- Inspect the available columns and data types;
- Identify missing values and duplicate records;
- Examine basic numerical and categorical characteristics;
- Identify potential data-quality issues;
- Create an initial data dictionary;
- Record issues that require further investigation or team decisions.

> **Scope:** This notebook is for understanding the raw dataset only. No cleaning, encoding or other transformations are performed here. The original dataset in `data/raw/` remains unchanged.

## 1. Load the Dataset

Pandas is used to load the original Airbnb CSV file into a DataFrame named `df`.

The raw dataset is read from `data/raw/` and is not modified in this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/AB_NYC_2019.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 2. Dataset Dimensions

The shape of the DataFrame is checked to determine the number of records and variables in the dataset.

In [2]:
df.shape

(48895, 16)

### Observation

The dataset contains **48,895 rows and 16 columns**.

- Each row represents an Airbnb listing record.
- Each column represents a characteristic or measurement associated with the listing.

## 3. Preview of the Dataset

The first five records are displayed to understand how the dataset is structured and to inspect examples of the values stored in each column.

In [3]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


### Initial Observation

The dataset contains different types of information, including:

- listing and host identifiers;
- listing and host names;
- geographical information;
- neighbourhood information;
- room type;
- price;
- minimum-night requirements;
- review information;
- host listing counts;
- annual availability.

A formal data dictionary is created later in this notebook.

## 4. Dataset Structure and Data Types

`df.info()` is used to inspect the column names, non-null values and Pandas data types.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

### Interpretation

The dataset contains:

- `int64` fields — whole-number numerical values;
- `float64` fields — numerical values that may contain decimals;
- `object` fields — mainly text or categorical information.

The non-null counts also indicate that some columns contain missing values. These are investigated separately below.

Some variables may require type conversion during later data preparation. No conversions are performed at this stage.

## 5. Missing Values

Missing values are checked to identify incomplete fields in the raw dataset.

In [5]:
df.isnull().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [6]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,missing_count,missing_percent
name,16,0.03
host_name,21,0.04
last_review,10052,20.56
reviews_per_month,10052,20.56


### Observation

Missing values are concentrated in four columns:

- `name`: **16** missing;
- `host_name`: **21** missing;
- `last_review`: **10,052** missing;
- `reviews_per_month`: **10,052** missing.

The identical missing counts for `last_review` and `reviews_per_month` suggest that these two review-related fields may be connected. This should be investigated before deciding how the missing values should be treated.

No missing values are present in the other columns.

> **Decision status:** No missing values are modified at this stage.

### Investigation of Missing Review Information

Both `last_review` and `reviews_per_month` contain 10,052 missing values.  
To better understand these missing values, the corresponding `number_of_reviews` values are examined.

This helps determine whether the missing review-related information is associated with listings that have received no reviews.

In [7]:
df.loc[
    df["reviews_per_month"].isnull(),
    ["number_of_reviews", "last_review", "reviews_per_month"]
].head(10)

,number_of_reviews,last_review,reviews_per_month
2,0,NaN,NaN
19,0,NaN,NaN
26,0,NaN,NaN
36,0,NaN,NaN
38,0,NaN,NaN
193,0,NaN,NaN
204,0,NaN,NaN
260,0,NaN,NaN
265,0,NaN,NaN
267,0,NaN,NaN


In [8]:
df.loc[
    df["last_review"].isnull(),
    "number_of_reviews"
].value_counts()

number_of_reviews
0    10052
Name: count, dtype: int64

#### Check the Number of Reviews for Missing `reviews_per_month`

The following check examines the number of reviews associated with listings where `reviews_per_month` is missing.

In [9]:
df.loc[
    df["reviews_per_month"].isnull(),
    "number_of_reviews"
].value_counts().head()

number_of_reviews
0    10052
Name: count, dtype: int64

### Observation

All **10,052 listings** with missing `reviews_per_month` have `number_of_reviews` equal to **0**.

The sample inspection also shows that `last_review` is missing for these listings. This indicates that the missing review-related values are associated with listings that have received no reviews, rather than appearing randomly across listings with review activity.

No values are changed at this stage. The appropriate treatment of these missing values will be decided during data preparation.

## 6. Duplicate and Listing ID Checks

The dataset is checked for completely duplicated rows and the uniqueness of the Airbnb listing identifier.

In [10]:
duplicate_count = df.duplicated().sum()

duplicate_count

np.int64(0)

In [11]:
unique_listing_ids = df["id"].nunique()

unique_listing_ids

48895

In [12]:
total_rows = len(df)

print("Total rows:", total_rows)
print("Unique listing IDs:", unique_listing_ids)
print("Duplicate rows:", duplicate_count)

Total rows: 48895
Unique listing IDs: 48895
Duplicate rows: 0


### Observation

- Complete duplicate rows: **0**
- Total records: **48,895**
- Unique listing IDs: **48,895**

Each record therefore has a unique listing ID, and no completely duplicated rows were identified.

## 7. Numerical Summary

A focused statistical summary is produced for numerical variables that are relevant to understanding listings and potential future modelling.

Identifier fields such as `id` and `host_id` are excluded because statistics such as their average do not provide meaningful business information.

In [13]:
numeric_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

In [14]:
numeric_summary = df[numeric_columns].describe().round(2)

numeric_summary

,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,48895.00,48895.00,48895.00,38843.00,48895.00,48895.00
mean,152.72,7.03,23.27,1.37,7.14,112.78
std,240.15,20.51,44.55,1.68,32.95,131.62
min,0.00,1.00,0.00,0.01,1.00,0.00
25%,69.00,1.00,1.00,0.19,1.00,0.00
50%,106.00,3.00,5.00,0.72,1.00,45.00
75%,175.00,5.00,24.00,2.02,2.00,227.00
max,10000.00,1250.00,629.00,58.50,327.00,365.00


## 8. Initial Data-Quality Checks

Several targeted checks are performed based on the numerical summary. These checks identify values that may require further investigation during EDA or data preparation.

In [15]:
zero_price_count = (df["price"] == 0).sum()

zero_price_count

np.int64(11)

### Zero-price Listings

There are 11 listings where `price` is 0.

These records are flagged for further investigation. They are not removed in this notebook.

In [16]:
df.nlargest(10, "price")[
    ["id", "neighbourhood_group", "room_type", "price"]
]

,id,neighbourhood_group,room_type,price
9151,7003697,Queens,Private room,10000
17692,13894339,Brooklyn,Entire home/apt,10000
29238,22436899,Manhattan,Entire home/apt,10000
6530,4737930,Manhattan,Entire home/apt,9999
12342,9528920,Manhattan,Private room,9999
40433,31340283,Manhattan,Entire home/apt,9999
30268,23377410,Manhattan,Entire home/apt,8500
4377,2953058,Brooklyn,Entire home/apt,8000
29662,22779726,Manhattan,Entire home/apt,7703
42523,33007610,Manhattan,Entire home/apt,7500


In [17]:
df.nlargest(10, "minimum_nights")[
    ["id", "neighbourhood_group", "room_type", "minimum_nights"]
]

,id,neighbourhood_group,room_type,minimum_nights
5767,4204302,Manhattan,Entire home/apt,1250
2854,1615764,Manhattan,Entire home/apt,1000
13404,10053943,Manhattan,Entire home/apt,999
26341,20990053,Brooklyn,Private room,999
38664,30378211,Manhattan,Shared room,999
7355,5431845,Queens,Entire home/apt,500
8014,6169897,Manhattan,Entire home/apt,500
11193,8668115,Brooklyn,Private room,500
14285,11096888,Brooklyn,Private room,500
47620,35834935,Brooklyn,Entire home/apt,500


In [18]:
print("Minimum availability:", df["availability_365"].min())
print("Maximum availability:", df["availability_365"].max())

Minimum availability: 0
Maximum availability: 365


## 9. Important Categorical Variables

Key categorical variables are inspected to understand the broad groups represented in the dataset.

In [19]:
df["neighbourhood_group"].value_counts()

neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

In [20]:
df["room_type"].value_counts()

room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64

In [21]:
df["neighbourhood"].nunique()

221

In [22]:
df["neighbourhood_group"].nunique()

5

In [23]:
df["room_type"].nunique()

3

## 10. Data Dictionary

A data dictionary is created to document the structure and meaning of each variable in the raw dataset.

In [24]:
data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notnull().sum().values,
    "missing_count": df.isnull().sum().values,
    "unique_values": df.nunique(dropna=True).values
})

data_dictionary

,column,dtype,non_null_count,missing_count,unique_values
0,id,int64,48895,0,48895
1,name,object,48879,16,47905
2,host_id,int64,48895,0,37457
3,host_name,object,48874,21,11452
4,neighbourhood_group,object,48895,0,5
5,neighbourhood,object,48895,0,221
6,latitude,float64,48895,0,19048
7,longitude,float64,48895,0,14718
8,room_type,object,48895,0,3
9,price,int64,48895,0,674


In [25]:
descriptions = {
    "id": "Unique Airbnb listing identifier",
    "name": "Listing name or title",
    "host_id": "Unique host identifier",
    "host_name": "Host name",
    "neighbourhood_group": "New York City borough",
    "neighbourhood": "Local neighbourhood",
    "latitude": "Geographical latitude of the listing",
    "longitude": "Geographical longitude of the listing",
    "room_type": "Type of accommodation",
    "price": "Listing price",
    "minimum_nights": "Minimum number of nights required",
    "number_of_reviews": "Total number of reviews",
    "last_review": "Date of the most recent review",
    "reviews_per_month": "Average number of reviews per month",
    "calculated_host_listings_count": "Number of listings associated with the host",
    "availability_365": "Number of days available within a year"
}

data_dictionary["description"] = (
    data_dictionary["column"].map(descriptions)
)

data_dictionary

,column,dtype,non_null_count,missing_count,unique_values,description
0,id,int64,48895,0,48895,Unique Airbnb listing identifier
1,name,object,48879,16,47905,Listing name or title
2,host_id,int64,48895,0,37457,Unique host identifier
3,host_name,object,48874,21,11452,Host name
4,neighbourhood_group,object,48895,0,5,New York City borough
5,neighbourhood,object,48895,0,221,Local neighbourhood
6,latitude,float64,48895,0,19048,Geographical latitude of the listing
7,longitude,float64,48895,0,14718,Geographical longitude of the listing
8,room_type,object,48895,0,3,Type of accommodation
9,price,int64,48895,0,674,Listing price


## 11. Data Quality Summary

The main structural and quality findings are summarised below for use by the team during EDA and data preparation.

In [26]:
quality_summary = pd.DataFrame({
    "check": [
        "Total rows",
        "Total columns",
        "Duplicate rows",
        "Unique listing IDs",
        "Missing name",
        "Missing host_name",
        "Missing last_review",
        "Missing reviews_per_month",
        "Zero-price listings",
        "Minimum availability",
        "Maximum availability"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df["id"].nunique(),
        df["name"].isnull().sum(),
        df["host_name"].isnull().sum(),
        df["last_review"].isnull().sum(),
        df["reviews_per_month"].isnull().sum(),
        (df["price"] == 0).sum(),
        df["availability_365"].min(),
        df["availability_365"].max()
    ]
})

quality_summary

,check,value
0,Total rows,48895
1,Total columns,16
2,Duplicate rows,0
3,Unique listing IDs,48895
4,Missing name,16
5,Missing host_name,21
6,Missing last_review,10052
7,Missing reviews_per_month,10052
8,Zero-price listings,11
9,Minimum availability,0


## 12. Save Data-Understanding Outputs

Reusable summary tables are saved under `outputs/tables/` so that they can be reviewed by other team members and reused later without modifying the raw dataset.

In [27]:
missing_summary.to_csv(
    "../outputs/tables/missing_values_summary.csv"
)

numeric_summary.to_csv(
    "../outputs/tables/numeric_summary.csv"
)

data_dictionary.to_csv(
    "../outputs/tables/data_dictionary.csv",
    index=False
)

quality_summary.to_csv(
    "../outputs/tables/data_quality_summary.csv",
    index=False
)

print("Data-understanding outputs saved successfully.")

Data-understanding outputs saved successfully.


# 13. Data Understanding Summary and Next Steps

The Airbnb NYC 2019 dataset contains **48,895 listings and 16 variables**.

Initial inspection identified **no completely duplicated rows**, and all **48,895 listing IDs are unique**.

Missing data is concentrated mainly in the review-related variables. `last_review` and `reviews_per_month` each contain **10,052 missing values (20.56%)**. A small number of missing values also occur in `name` (16) and `host_name` (21).

Initial numerical checks identified **11 zero-price listings** and potentially extreme values in both `price` and `minimum_nights`. These records have not been removed because further investigation is required before deciding whether they represent genuine observations or data-quality problems.

The dataset contains **5 neighbourhood groups, 221 neighbourhoods and 3 room types**, providing useful categorical information for subsequent analysis.

For the later regression task, `price` is the potential target variable. Candidate explanatory variables include location, room type, minimum nights, reviews, availability and host listing count. Final feature selection will be decided after EDA and data preparation.

## Next Steps

The next stage is `02_eda.ipynb`, where price distributions, neighbourhoods, room types, reviews, availability and relationships between relevant variables will be explored visually and statistically.